#  Order Items - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, DecimalType, TimestampType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_order_items"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "order_items"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DecimalType(8, 2), True),
    StructField("freight_value", DecimalType(8, 2), True),

])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(8,2) (nullable = true)
 |-- freight_value: decimal(8,2) (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_items/olist_order_items_dataset.csv,2026-08-02T21:28:52.000Z,2026-08-02T22:41:27.544Z,1c48f958-bab4-4da9-a7bb-f1a2f79c92c8,olist,order_items
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_items/olist_order_items_dataset.csv,2026-08-02T21:28:52.000Z,2026-08-02T22:41:27.544Z,1c48f958-bab4-4da9-a7bb-f1a2f79c92c8,olist,order_items
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_items/olist_order_items_dataset.csv,2026-08-02T21:28:52.000Z,2026-08-02T22:41:27.544Z,1c48f958-bab4-4da9-a7bb-f1a2f79c92c8,olist,order_items
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_items/olist_order_items_dataset.csv,2026-08-02T21:28:52.000Z,2026-08-02T22:41:27.544Z,1c48f958-bab4-4da9-a7bb-f1a2f79c92c8,olist,order_items
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_items/olist_order_items_dataset.csv,2026-08-02T21:28:52.000Z,2026-08-02T22:41:27.544Z,1c48f958-bab4-4da9-a7bb-f1a2f79c92c8,olist,order_items


In [0]:
spark.table(target_table).count()

112650